In [1]:
import llava
import transformers
import json

/home/yifayang/Documents/Projects/Other/LLaVA/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-08-02 18:07:47,555] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [2]:
proj_dir = "/home/yifayang/Documents/Projects/Other/LLaVA"

class ModelArguments:
    model_name_or_path = f"{proj_dir}/playground/checkpoints/vicuna-7b-v1.3"

class TrainingArguments:
    cache_dir = "/home/yifayang/.cache/huggingface"
    model_max_length = 2048

class DataArguments:
    data_path = f"{proj_dir}/playground/data/pretrain_cc3m_558k/chat.json"

model_args = ModelArguments()
training_args = TrainingArguments()
data_args = DataArguments()

In [3]:
tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_args.model_name_or_path,
    cache_dir=training_args.cache_dir,
    model_max_length=training_args.model_max_length,
    padding_side="right",
    use_fast=False,
)

You are using the legacy behaviour of the <class 'transformers.models.llama.tokenization_llama.LlamaTokenizer'>. This means that tokens that come after special tokens will not be properly handled. We recommend you to read the related pull request available at https://github.com/huggingface/transformers/pull/24565


In [4]:
list_data_dict = json.load(open(f"{proj_dir}/playground/data/pretrain_cc3m_558k/chat.json", "r"))

In [5]:
list_data_dict[0]

{'id': 'GCC_train_002582585',
 'image': 'GCC_train_002582585.jpg',
 'conversations': [{'from': 'human',
   'value': 'Provide a brief description of the given image.\n<image>'},
  {'from': 'gpt',
   'value': 'olive oil is a healthy ingredient used liberally .'}]}

In [6]:
list_data_dict[1]

{'id': 'GCC_train_002429825',
 'image': 'GCC_train_002429825.jpg',
 'conversations': [{'from': 'human',
   'value': '<image>\nWrite a terse but informative summary of the picture.'},
  {'from': 'gpt',
   'value': '3d vector deluxe alphabet of randomly rotated thin golden symbols .'}]}

In [7]:
list_data_dict[3]

{'id': 'GCC_train_002503829',
 'image': 'GCC_train_002503829.jpg',
 'conversations': [{'from': 'human',
   'value': 'Relay a brief, clear account of the picture shown.\n<image>'},
  {'from': 'gpt',
   'value': 'fans interferes with # on a ball hit by # of sports team scoring runs in the eighth inning during game'}]}

In [8]:
list_data_dict[4]

{'id': 'GCC_train_000530863',
 'image': 'GCC_train_000530863.jpg',
 'conversations': [{'from': 'human',
   'value': '<image>\nRender a clear and concise summary of the photo.'},
  {'from': 'gpt', 'value': 'train that takes you around the perimeter'}]}

Default
```bash
python llava/train/train_mem.py \
    --model_name_or_path ./playground/checkpoints/vicuna-7b-v1.3 \
    --version v1 \
    --data_path ./playground/data/pretrain_cc3m_558k/chat.json \
    --image_folder ./playground/data/pretrain_cc3m_558k \
    --vision_tower openai/clip-vit-large-patch14 \
    --tune_mm_mlp_adapter True \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --bf16 True \
    --output_dir ./playground/checkpoints/llava-vicuna-7b-v1.3-pretrain \
    --num_train_epochs 1 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 2400 \
    --save_total_limit 1 \
    --learning_rate 2e-3 \
    --weight_decay 0. \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --lazy_preprocess True \
    --report_to wandb
```

Option A (recommended): use the `deepspeed` launcher so it sets the

distributed env vars and skips MPI discovery. `--include localhost:1`

picks GPU 1 (do NOT also set CUDA_VISIBLE_DEVICES here).
```bash
deepspeed --include localhost:0 llava/train/train_mem.py \
    --model_name_or_path ./playground/checkpoints/vicuna-7b-v1.3 \
    --version v1 \
    --data_path ./playground/data/pretrain_cc3m_558k/chat.json \
    --image_folder ./playground/data/pretrain_cc3m_558k \
    --vision_tower openai/clip-vit-large-patch14 \
    --tune_mm_mlp_adapter True \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --bf16 True \
    --output_dir ./playground/checkpoints/llava-vicuna-7b-v1.3-pretrain \
    --num_train_epochs 1 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 2400 \
    --save_total_limit 1 \
    --learning_rate 2e-3 \
    --weight_decay 0. \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --lazy_preprocess True \
    --deepspeed ./scripts/zero3.json \
    --report_to wandb
```

Option B: keep plain `python` but set the distributed env vars yourself
so DeepSpeed does not fall back to MPI discovery (no mpi4py needed).
```bash
CUDA_VISIBLE_DEVICES=0 \
RANK=0 LOCAL_RANK=0 WORLD_SIZE=1 MASTER_ADDR=localhost MASTER_PORT=29500 \
python llava/train/train_mem.py \
    --model_name_or_path ./playground/checkpoints/vicuna-7b-v1.3 \
    --version v1 \
    --data_path ./playground/data/pretrain_cc3m_558k/chat.json \
    --image_folder ./playground/data/pretrain_cc3m_558k \
    --vision_tower openai/clip-vit-large-patch14 \
    --tune_mm_mlp_adapter True \
    --mm_vision_select_layer -2 \
    --mm_use_im_start_end False \
    --mm_use_im_patch_token False \
    --bf16 True \
    --output_dir ./playground/checkpoints/llava-vicuna-7b-v1.3-pretrain \
    --num_train_epochs 1 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 4 \
    --gradient_accumulation_steps 8 \
    --evaluation_strategy "no" \
    --save_strategy "steps" \
    --save_steps 2400 \
    --save_total_limit 1 \
    --learning_rate 2e-3 \
    --weight_decay 0. \
    --warmup_ratio 0.03 \
    --lr_scheduler_type "cosine" \
    --logging_steps 1 \
    --tf32 True \
    --model_max_length 2048 \
    --gradient_checkpointing True \
    --lazy_preprocess True \
    --report_to wandb
```